# Run Helpers with Local or Privacy-Preserving LLMs

Till now, we’ve been using Google’s Gemini models, which means our prompts and data are sent through an external API.

In this lesson, we switch to the same helper pattern but run it with a local Ollama model. This matters because not all data can leave your environment. Running models locally keeps everything on your own machine, which is important when you’re dealing with private, sensitive, or restricted data.

## 1 — Setup

We keep setup simple: install libraries, import them, configure Ollama, and load the dataset.

This notebook uses a local Ollama model instead of a cloud API key. To run it, install Ollama, start the Ollama server, and pull the model named in this setup section.


### 1.1 — About Ollama (Local-First)

Ollama lets you run open-weight models locally on your machine. For this lesson, we use only local inference (`localhost:11434`) and no cloud models.


### 1.2 — Ollama Setup 

1. Install from https://ollama.com/download
2. Verify: `ollama --version`
3. Start server: `ollama serve`
4. Pull model: `ollama pull qwen2.5-coder:7b`
5. Quick test: `ollama run qwen2.5-coder:7b`


In [ ]:
%pip install -qq pandas ollama python-dotenv

In [ ]:

import re
import ollama
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

# Base URL of the local Ollama server. Override via OLLAMA_URL env var if needed.
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5-coder:7b"

# Create a client pointing to the Ollama server URL
client = ollama.Client(host=OLLAMA_URL)

print("OLLAMA_URL:", OLLAMA_URL)
print("OLLAMA_MODEL:", OLLAMA_MODEL)

In [ ]:
df = pd.read_csv("../../data/hr_analytics.csv")
print(df.shape)

## 2 — Local LLM Helper Function

Same helper contract: model returns pandas code that writes final output to `result_df`.


In [ ]:
SYSTEM_PROMPT = (
    "Write pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. Avoid deprecated arguments or methods. "
    "Store the final result in `result_df`. "
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations. "
    "If a column contains Yes/No strings and you need to compute a rate or mean, "
    "convert it first with .eq('Yes').astype(int) before aggregating."
)


def eda_helper_local(question: str, frame: pd.DataFrame, show_code: bool = False):
    # 1. Send the question + column names to the local Ollama model.
    #    The response is raw text — the model's reply before we extract the code.
    text = client.generate(
        model=OLLAMA_MODEL,
        prompt=f"Columns: {list(frame.columns)}\nQuestion: {question}",
        system=SYSTEM_PROMPT,
    ).response

    # 2. Extract the code block from the model response
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code_text = match.group(1).strip() if match else text.strip()

    if show_code:
        print("--- Generated Code (Local) ---")
        print(code_text)
        print("---")

    # 3. Run the generated code in a sandboxed environment
    env = {"pd": pd, "df": frame.copy()}
    try:
        exec(code_text, env, env)
    except Exception as e:
        print("--- Generated Code (caused error) ---")
        print(code_text)
        print("---")
        raise RuntimeError(f"Generated code failed: {e}") from e

    # 4. Return the result the model stored in result_df
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result

## 3 — Running EDA helper with Ollama instead of a cloud hosted LLM

This lesson focuses on inspecting local-model-generated code before trusting outputs.


In [ ]:
eda_helper_local(
    "How many missing values are there in each column?", df, show_code=True
)